# EMEA GTM Intelligence Engine
**Category Business Manager Portfolio Project**

This notebook mirrors the analytical workflow a Category Business Manager executes to:
1. Size the EMEA premium smartphone TAM by country
2. Identify upgrader/switcher demand pools
3. Model promotional ROI across channels
4. Build a composite Demand Signal Index to prioritise market investment
5. Track competitive share shifts and derive GTM recommendations

**Data Sources (Kaggle)**
- [Global Electronics Retailer Dataset](https://www.kaggle.com/datasets/bhavikjikadara/global-electronics-retailer)
- [Consumer Electronics Market Analysis](https://www.kaggle.com/datasets/manus25/consumer-electronics-market-analysis)
- [World GDP & Population](https://www.kaggle.com/datasets/tunguz/country-regional-and-world-gdp)

> **Resume signal**: This project demonstrates skills directly mapped to the Google Category Business Manager JD: data-led TAM analysis, GTM strategy development, promo investment modelling, demand signal interpretation, and cross-market prioritisation.

In [ ]:
# Cell 1 – Environment setup
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'matplotlib', 'seaborn', 'plotly', '-q'])

import json, math, random
from collections import defaultdict
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('Environment ready')

In [ ]:
# Cell 2 – Load pre-computed outputs from analysis pipeline
# If running from scratch, execute: python src/emea_gtm_analysis.py first

import os
OUTPUTS = 'outputs'

def load(name):
    path = os.path.join(OUTPUTS, f'{name}.json')
    with open(path) as f:
        return json.load(f)

tam_df   = pd.DataFrame(load('tam_sizing'))
perf_df  = pd.DataFrame(load('quarterly_performance'))
comp_df  = pd.DataFrame(load('competitive_landscape'))
promo_df = pd.DataFrame(load('promotional_roi'))
fun_df   = pd.DataFrame(load('switcher_upgrader_funnel'))
dsi_df   = pd.DataFrame(load('demand_signal_index'))
summary  = load('executive_summary')

print(f"Loaded {len(tam_df)} markets | {len(comp_df)} competitive rows | {len(perf_df)} perf records")

## 1. Executive Summary
High-level KPIs surfaced from the analysis pipeline.

In [ ]:
# Cell 3 – Executive KPI summary
kpis = pd.Series(summary)
print('═' * 52)
print('  EMEA GTM Intelligence — Executive Summary')
print('═' * 52)
for k, v in summary.items():
    label = k.replace('_', ' ').title()
    print(f'  {label:<35} {v}')
print('═' * 52)

## 2. TAM Sizing & Market Prioritisation
**Methodology**: Population × Smartphone Penetration × Premium Share × Average Selling Price ($650 premium ASP)

**Key insight**: Germany and UK together represent ~36% of EMEA TAM. UAE and Sweden show disproportionate premium-share upside relative to population.

In [ ]:
# Cell 4 – TAM waterfall by country
top8 = tam_df.head(8).copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# TAM vs current revenue
ax = axes[0]
x = range(len(top8))
ax.bar(x, top8['tam_mn_usd']/1000, color='#B5D4F4', label='TAM ($B)', width=0.6)
ax.bar(x, top8['current_rev_mn_usd']/1000, color='#185FA5', label='Current rev ($B)', width=0.6)
ax.set_xticks(x)
ax.set_xticklabels(top8['country'], rotation=30, ha='right', fontsize=9)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'${v:.0f}B'))
ax.set_title('TAM vs Current Revenue', fontweight='bold')
ax.legend(fontsize=9)

# Growth opportunity index scatter
ax2 = axes[1]
colors = {'low': '#1D9E75', 'medium': '#EF9F27', 'high': '#E24B4A'}
for _, row in tam_df.iterrows():
    ax2.scatter(row['premium_share_pct'], row['growth_opportunity_index'],
                s=row['tam_mn_usd']/60, alpha=0.75,
                color=colors[row['fx_risk']], edgecolors='white', linewidths=0.5)
    ax2.annotate(row['country'], (row['premium_share_pct'], row['growth_opportunity_index']),
                 fontsize=7.5, ha='center', va='bottom')
ax2.set_xlabel('Premium segment share (%)')
ax2.set_ylabel('Growth opportunity index')
ax2.set_title('Opportunity Matrix — bubble = TAM size', fontweight='bold')
# Custom legend for FX risk
for label, col in colors.items():
    ax2.scatter([], [], color=col, label=f'FX: {label}', s=60)
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('outputs/tam_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to outputs/tam_analysis.png')

## 3. Competitive Share Analysis
**Insight**: Strong share gains driven by AI-differentiated camera features from H2 2023, primarily capturing upgrader volume from mid-tier Android incumbents.

In [ ]:
# Cell 5 – Competitive share trends
pivot = comp_df.pivot(index='quarter', columns='brand', values='emea_share_pct')

fig, ax = plt.subplots(figsize=(13, 5))
colors_brand = {'Pixel':'#185FA5','Samsung':'#888780','Apple':'#1D9E75',
                'Xiaomi':'#D85A30','Other':'#D4537E'}
lw = {'Pixel': 2.5}

for brand in pivot.columns:
    ax.plot(pivot.index, pivot[brand], label=brand,
            color=colors_brand.get(brand,'#888'),
            linewidth=lw.get(brand, 1.5),
            marker='o' if brand == 'Pixel' else None,
            markersize=5)

ax.set_xticklabels(pivot.index, rotation=45, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda v, _: f'{v:.0f}%'))
ax.set_title('EMEA Premium Smartphone Market Share 2022–2024', fontweight='bold')
ax.legend(loc='upper left', fontsize=9)
ax.axvline(x=4, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)
ax.text(4.1, 35, 'AI camera launch ▶', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('outputs/competitive_share.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Promotional ROI Modelling
**Methodology**: Investment × price elasticity → incremental unit volume → incremental revenue. ROI = (Incr. Rev − Investment) / Investment.

**Recommendation**: Cashback and trade-in via Open Market channel deliver best ROI with contained margin compression. Avoid Early Bird discounting — elasticity insufficient to justify.

In [ ]:
# Cell 6 – Promo ROI scatter + table
fig, ax = plt.subplots(figsize=(10, 5))

for _, row in promo_df.iterrows():
    color = '#1D9E75' if row['roi_pct'] > 70 else ('#185FA5' if row['roi_pct'] > 0 else '#E24B4A')
    ax.scatter(row['margin_compression_pct'], row['roi_pct'],
               s=row['invest_mn'] * 20, alpha=0.75, color=color, edgecolors='white')
    ax.annotate(f"{row['promo']}\n{row['channel']}",
                (row['margin_compression_pct'], row['roi_pct']),
                fontsize=7, ha='center', va='bottom')

ax.axhline(0, color='red', linewidth=0.7, linestyle='--')
ax.axhline(70, color='green', linewidth=0.7, linestyle='--', alpha=0.5)
ax.set_xlabel('Margin compression (%)')
ax.set_ylabel('ROI (%)')
ax.set_title('Promo ROI vs Margin Compression (bubble = investment size)', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/promo_roi.png', dpi=150, bbox_inches='tight')
plt.show()

print(promo_df[['promo','channel','invest_mn','roi_pct','margin_compression_pct']]
      .sort_values('roi_pct', ascending=False).to_string(index=False))

## 5. Demand Signal Index
Composite index (0–100) combining search trend proxy, sell-out velocity, retailer stock turns, and NPS proxy. Markets are classified as **Accelerate** (>65), **Sustain** (45–65), or **Build** (<45).

In [ ]:
# Cell 7 – DSI horizontal bar chart
dsi_sorted = dsi_df.sort_values('demand_signal_index', ascending=True)
colors_sig = {'Accelerate':'#1D9E75','Sustain':'#185FA5','Build':'#D85A30'}

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(dsi_sorted['country'],
               dsi_sorted['demand_signal_index'],
               color=[colors_sig[s] for s in dsi_sorted['signal']],
               edgecolor='white', height=0.65)

for bar, val in zip(bars, dsi_sorted['demand_signal_index']):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)

ax.axvline(65, color='#1D9E75', linestyle='--', linewidth=0.8, alpha=0.6)
ax.axvline(45, color='#D85A30', linestyle='--', linewidth=0.8, alpha=0.6)
ax.set_xlim(0, 105)
ax.set_xlabel('Demand Signal Index')
ax.set_title('EMEA Demand Signal Index — Market Investment Signal', fontweight='bold')

for label, col in colors_sig.items():
    ax.barh([], [], color=col, label=label)
ax.legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('outputs/demand_signal.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Switcher & Upgrader Funnel
Decision funnel from awareness → consideration → intent → conversion across three demand segments.

In [ ]:
# Cell 8 – Funnel visualisation
stages = ['awareness_pct','consideration_pct','intent_pct','conversion_pct']
labels = ['Awareness','Consideration','Intent','Conversion']
seg_colors = {'Upgrader (Android)':'#185FA5','Switcher (iOS)':'#D85A30','New-to-premium':'#1D9E75'}

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, (_, row) in zip(axes, fun_df.iterrows()):
    values = [row[s] for s in stages]
    col = seg_colors[row['segment']]
    ax.barh(labels[::-1], values[::-1], color=col, alpha=0.8, edgecolor='white')
    for i, v in enumerate(values[::-1]):
        ax.text(v + 0.5, i, f'{v:.0f}%', va='center', fontsize=9)
    ax.set_xlim(0, 100)
    ax.set_title(row['segment'], fontsize=10, fontweight='bold', color=col)
    ax.set_xlabel('% of segment')
    ax.text(50, -0.7, f"Pool: {row['addressable_pool_m']}M | Buyers: {int(row['projected_buyers_k'])}K",
            ha='center', fontsize=8, color='gray')

plt.suptitle('Switcher & Upgrader Decision Funnel — EMEA', fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/funnel.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. GTM Recommendations

| Priority | Market cluster | Action | Rationale |
|---|---|---|---|
| 1 | UK, Germany, France | Accelerate carrier trade-in investment | DSI >80, Tier 1 TAM, proven elasticity |
| 2 | UAE, Netherlands | Premium bundling via e-tailer | High premium share, FX stable |
| 3 | Spain, Italy | Upgrader re-targeting at MoT | Mid-tier Android upgrader pool |
| 4 | Poland | Build brand awareness | Low DSI but growing premium appetite |
| 5 | Turkey, South Africa | Monitor — FX headwind | High risk, wait for stabilisation |

**Biggest funnel gap**: Upgrader segment drops 21pp from Awareness → Consideration. Invest in comparison content and in-store trials at Tier 1 retail.

**Promo recommendation**: Cashback (Open Market) and Trade-in (Carrier) deliver ROI >100% with margin compression <7%. Prioritise these in H2 launch window.